# Stock Price Prediction Pipeline (TPU Optimized)

## Phase 3 Ensemble Model (Stock-wise Independent Training)

### Prediction Setup
- **Predict**: Close(t)
- **Using Data Till**: (t-1)
- **Include**: Open(t)

### Key Guarantees
- ✓ No data leakage
- ✓ Sequential time-series split
- ✓ TPU optimized
- ✓ Heavyweight Transformer + LSTM + Dense Ensemble

In [ ]:
# ============================================
# IMPORTS: All Required Libraries
# ============================================

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from optuna.samplers import TPESampler
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import json
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✓ All libraries imported successfully")
print(f"TensorFlow Version: {tf.__version__}")

## TPU Setup and Strategy

### TPU Requirements
- Requires **static tensor shapes** for optimal compilation
- `drop_remainder=True` ensures consistent batch sizes
- `bfloat16` dtype reduces memory footprint
- Mixed precision training improves training speed by 2-3x

### Strategy Pattern
- Creates `tf.distribute.TPUStrategy` for distributed training
- All models built and trained within strategy scope
- Automatic gradient synchronization across cores

In [ ]:
# ============================================
# TPU SETUP
# ============================================

# Detect TPU (will work on Kaggle, gracefully fallback to GPU/CPU locally)
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("✓ TPU initialized successfully")
    print(f"  Number of TPU cores: {strategy.num_replicas_in_sync}")
except ValueError:
    # Fallback to GPU or CPU
    print("⚠ TPU not detected. Using GPU/CPU strategy.")
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        strategy = tf.distribute.MirroredStrategy()
        print(f"✓ GPU strategy initialized with {len(gpus)} GPU(s)")
    else:
        strategy = tf.distribute.OneDeviceStrategy('/CPU:0')
        print("⚠ Using CPU strategy")

# Set global dtype to bfloat16 for TPU efficiency
keras.mixed_precision.set_global_policy('mixed_bfloat16')
print("\n✓ Global dtype policy set to bfloat16")
print(f"  Compute dtype: {keras.mixed_precision.global_policy().compute_dtype}")
print(f"  Variable dtype: {keras.mixed_precision.global_policy().variable_dtype}")

## Data Loading

### Process
1. Load Excel file from Kaggle input
2. Read all sheets (one stock per sheet)
3. Store in dictionary for processing
4. Display available stock symbols

In [ ]:
# ============================================
# DATA LOADING
# ============================================

# Define file paths
INPUT_FILE = '/kaggle/input/nse-stock-data/nse_stock_data.xlsx'
OUTPUT_DIR = '/kaggle/working'
MODELS_DIR = os.path.join(OUTPUT_DIR, 'Saved_Models')
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')

# Create output directories
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Load all sheets
print(f"Loading data from: {INPUT_FILE}")
stocks_data = pd.read_excel(INPUT_FILE, sheet_name=None)

print(f"\n✓ Loaded {len(stocks_data)} stock sheets")
print("\nAvailable stocks:")
for i, stock_name in enumerate(list(stocks_data.keys())[:10], 1):
    print(f"  {i}. {stock_name}")
if len(stocks_data) > 10:
    print(f"  ... and {len(stocks_data) - 10} more stocks")

stock_names = list(stocks_data.keys())
print(f"\nTotal stocks to process: {len(stock_names)}")

## Preprocessing and Data Cleaning

### Steps
1. **Detect date column** automatically
2. **Convert to datetime** with proper error handling
3. **Sort sequentially** by date (ascending)
4. **Remove duplicates** and invalid rows
5. **Add stock identifier** for tracking

In [ ]:
# ============================================
# PREPROCESSING & CLEANING
# ============================================

def detect_date_column(df):
    """Automatically detect date column"""
    date_variants = ['date', 'Date', 'DATE', 'datetime', 'Datetime', 'time', 'Time']
    
    for col in date_variants:
        if col in df.columns:
            return col
    
    for col in df.columns:
        if 'date' in col.lower() or 'time' in col.lower():
            return col
    
    return None

def clean_stock_data(df, stock_name):
    """Clean and prepare stock data"""
    df = df.copy()
    
    # Detect and convert date column
    date_col = detect_date_column(df)
    if date_col:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df = df.dropna(subset=[date_col])
        df = df.sort_values(by=date_col).reset_index(drop=True)
    else:
        print(f"  Warning: No date column detected in {stock_name}")
    
    # Add symbol column
    df['Symbol'] = stock_name
    
    return df

# Clean all stock data
print("Cleaning and preprocessing data...\n")
cleaned_stocks = {}
for stock_name, df in stocks_data.items():
    cleaned_stocks[stock_name] = clean_stock_data(df, stock_name)
    print(f"  ✓ {stock_name}: {len(cleaned_stocks[stock_name])} rows")

print(f"\n✓ Preprocessing completed for {len(cleaned_stocks)} stocks")

## Feature Engineering (Strict Time Logic)

### No Data Leakage Guarantee
- **Input features**: All data from time (t-1) and earlier
- **Include**: Open(t) (known at market open)
- **Exclude**: Close(t-1), High(t-1), Low(t-1), Volume(t-1) if not explicitly allowed
- **Target**: Close(t) (what we predict)

### Feature Strategy
- Lag features created with `.shift(1)` (ensures causality)
- All indicators computed on previous day data only
- Drop rows with NaN (warm-up period)

In [ ]:
# ============================================
# FEATURE ENGINEERING
# ============================================

def engineer_features(df):
    """Create lag and technical features (no leakage)"""
    df = df.copy()
    
    # Identify numeric OHLCV columns (case-insensitive)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Remove symbol and date columns from processing
    numeric_cols = [col for col in numeric_cols if col.lower() not in ['symbol']]
    
    # Create lag features: shift by 1 to avoid leakage
    for col in numeric_cols:
        df[f'{col}_lag1'] = df[col].shift(1)
        df[f'{col}_lag2'] = df[col].shift(2)
        df[f'{col}_lag3'] = df[col].shift(3)
    
    # Remove original OHLCV columns except Open(t) and Close(t) for target
    # Keep: Date, Open(t), Close(t), Symbol
    # Drop original features (we have lagged versions)
    for col in numeric_cols:
        if col.lower() not in ['open', 'close', 'volume']:
            if col in df.columns:
                df = df.drop(columns=[col])
    
    # Drop rows with NaN (warm-up period)
    df = df.dropna().reset_index(drop=True)
    
    return df

print("Feature engineering...\n")
feature_stocks = {}
for stock_name, df in cleaned_stocks.items():
    feature_stocks[stock_name] = engineer_features(df)
    print(f"  ✓ {stock_name}: {len(feature_stocks[stock_name])} rows, {len(feature_stocks[stock_name].columns)} features")

print(f"\n✓ Feature engineering completed")

## Sequence Building

### Purpose
- Convert tabular time-series into fixed-length sequences
- Enables RNN/Transformer training
- Maintains temporal dependencies

### Configuration
- **Sequence length**: 20 (20 trading days ≈ 1 month)
- **Sliding window**: 1 step (overlapping sequences)
- **Output shape**: (batch, sequence_len, features)

In [ ]:
# ============================================
# SEQUENCE BUILDING
# ============================================

SEQ_LEN = 20  # 20 trading days

def build_sequences(df, seq_len=SEQ_LEN):
    """
    Convert tabular data into fixed-length sequences.
    
    Returns:
        X: sequences of shape (n_samples, seq_len, n_features)
        y: targets of shape (n_samples,)
    """
    # Extract features (exclude date, symbol, and target)
    date_col = detect_date_column(df)
    cols_to_drop = ['Symbol']
    if date_col:
        cols_to_drop.append(date_col)
    
    # Get feature columns
    feature_cols = [col for col in df.columns if col not in cols_to_drop + ['Close']]
    
    X_data = df[feature_cols].values.astype(np.float32)
    y_data = df['Close'].values.astype(np.float32)
    
    X_seq = []
    y_seq = []
    
    for i in range(len(df) - seq_len):
        X_seq.append(X_data[i:i+seq_len])
        y_seq.append(y_data[i+seq_len])
    
    X_seq = np.array(X_seq, dtype=np.float32)
    y_seq = np.array(y_seq, dtype=np.float32)
    
    return X_seq, y_seq, feature_cols

print("Building sequences...\n")
sequence_data = {}
for stock_name, df in feature_stocks.items():
    if len(df) > SEQ_LEN:
        X, y, features = build_sequences(df, SEQ_LEN)
        sequence_data[stock_name] = {'X': X, 'y': y, 'features': features}
        print(f"  ✓ {stock_name}: X shape {X.shape}, y shape {y.shape}")
    else:
        print(f"  ⚠ {stock_name}: Insufficient data (need > {SEQ_LEN})")

print(f"\n✓ Sequences built for {len(sequence_data)} stocks")

## Train-Validation-Test Split (Sequential)

### Time-Based Splitting (No Random Shuffle)
- Preserves temporal order
- Prevents look-ahead bias
- Realistic for time-series

### Split Ratios
- **Train**: 70% (earliest data)
- **Validation**: 15% (middle data)
- **Test**: 15% (latest data - true future)

### Rationale
- Model trains on historical data
- Validated on recent data
- Tested on truly held-out future

In [ ]:
# ============================================
# TRAIN-VALIDATION-TEST SPLIT
# ============================================

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

def sequential_split(X, y, train_ratio=0.7, val_ratio=0.15):
    """
    Sequential time-based split (no shuffle).
    """
    n = len(X)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    X_train = X[:train_end]
    y_train = y[:train_end]
    
    X_val = X[train_end:val_end]
    y_val = y[train_end:val_end]
    
    X_test = X[val_end:]
    y_test = y[val_end:]
    
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

print("Performing sequential train-val-test split...\n")
split_data = {}
for stock_name, data in sequence_data.items():
    X, y = data['X'], data['y']
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = sequential_split(X, y, TRAIN_RATIO, VAL_RATIO)
    
    split_data[stock_name] = {
        'train': (X_train, y_train),
        'val': (X_val, y_val),
        'test': (X_test, y_test),
        'features': data['features']
    }
    
    print(f"  {stock_name}:")
    print(f"    Train: {X_train.shape} -> {y_train.shape}")
    print(f"    Val:   {X_val.shape} -> {y_val.shape}")
    print(f"    Test:  {X_test.shape} -> {y_test.shape}")

print(f"\n✓ Sequential split completed for {len(split_data)} stocks")

## TPU Data Pipeline (tf.data)

### Optimization Goals
- **Prevent TPU starvation** with prefetching
- **Cache data** in memory for faster iteration
- **Shuffle training data** for better generalization
- **Fixed batch sizes** required by TPU

### Pipeline Stages
1. `.from_tensor_slices()`: Create dataset from arrays
2. `.cache()`: Load into memory
3. `.shuffle()`: Random order (train only)
4. `.batch(128, drop_remainder=True)`: Fixed size batches
5. `.prefetch()`: Overlap computation and data loading

In [ ]:
# ============================================
# TPU DATA PIPELINE
# ============================================

BATCH_SIZE = 128
SHUFFLE_BUFFER = 1024

def create_tf_dataset(X, y, batch_size=128, shuffle=True):
    """
    Create optimized tf.data.Dataset for TPU.
    
    Args:
        X: Feature array
        y: Target array
        batch_size: Must be multiple of 128 for TPU
        shuffle: Whether to shuffle (True for train, False for val/test)
    
    Returns:
        Optimized tf.data.Dataset with static shapes
    """
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=SHUFFLE_BUFFER)
    
    dataset = dataset.cache()
    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

print("Creating tf.data pipelines...\n")
data_pipelines = {}
for stock_name, data in split_data.items():
    X_train, y_train = data['train']
    X_val, y_val = data['val']
    X_test, y_test = data['test']
    
    ds_train = create_tf_dataset(X_train, y_train, BATCH_SIZE, shuffle=True)
    ds_val = create_tf_dataset(X_val, y_val, BATCH_SIZE, shuffle=False)
    ds_test = create_tf_dataset(X_test, y_test, BATCH_SIZE, shuffle=False)
    
    data_pipelines[stock_name] = {
        'train': ds_train,
        'val': ds_val,
        'test': ds_test
    }
    
    print(f"  ✓ {stock_name}: Train {ds_train.element_spec}, Val {ds_val.element_spec}")

print(f"\n✓ Data pipelines created for {len(data_pipelines)} stocks")

## Model Architectures

### 1. Dense Neural Network
- Simple, fast baseline
- Flatten sequences
- Multiple dense layers with dropout

### 2. LSTM Model
- Captures temporal dependencies
- Multiple LSTM layers
- Dense output layer

### 3. Heavyweight Transformer
- Multi-head self-attention
- Positional encoding
- Stacked transformer blocks
- Residual connections
- Layer normalization

In [ ]:
# ============================================
# MODEL ARCHITECTURES
# ============================================

def build_dense_model(input_shape, hp=None):
    """
    Dense neural network baseline model.
    """
    model = models.Sequential([
        layers.Flatten(input_shape=input_shape),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(1)
    ])
    return model

def build_lstm_model(input_shape, hp=None):
    """
    LSTM model for sequential processing.
    """
    model = models.Sequential([
        layers.LSTM(128, return_sequences=True, input_shape=input_shape),
        layers.Dropout(0.2),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32, return_sequences=False),
        layers.Dropout(0.1),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    return model

def positional_encoding(length, depth, dtype=tf.float32):
    """
    Positional encoding for Transformer.
    """
    depth = depth / 2
    positions = np.arange(length)[:, np.newaxis]
    depths = np.arange(depth)[np.newaxis, :] / depth
    
    angle_rates = 1 / (10000 ** depths)
    angle_rads = positions * angle_rates
    
    pos_encoding = np.concatenate([np.sin(angle_rads), np.cos(angle_rads)], axis=-1)
    return tf.cast(pos_encoding[np.newaxis, ...], dtype=dtype)

def build_transformer_model(input_shape, hp=None):
    """
    Heavyweight Transformer model with multi-head attention.
    """
    seq_len, n_features = input_shape
    
    inputs = layers.Input(shape=input_shape)
    x = inputs
    
    # Positional encoding
    pos_enc = positional_encoding(seq_len, n_features)
    x = x + pos_enc
    
    # Transformer blocks
    n_heads = 8
    d_model = 64
    
    for _ in range(3):  # 3 transformer layers
        # Multi-head attention
        attn_output = layers.MultiHeadAttention(
            num_heads=n_heads, 
            key_dim=d_model // n_heads
        )(x, x)
        x = layers.Add()([x, attn_output])
        x = layers.LayerNormalization(epsilon=1e-6)(x)
        
        # Feed-forward
        ff = layers.Dense(256, activation='relu')(x)
        ff = layers.Dense(d_model)(ff)
        x = layers.Add()([x, ff])
        x = layers.LayerNormalization(epsilon=1e-6)(x)
    
    # Output
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(1)(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

print("✓ Model architectures defined")

## Optuna Hyperparameter Tuning

### Optimization Objective
- Minimize validation MSE
- Search over transformer and model parameters

### Tunable Hyperparameters
- Learning rate: 1e-5 to 1e-2
- Number of heads: 4, 8, 16
- Number of layers: 1-4
- Dropout rate: 0.1-0.5
- Hidden size: 32-256

In [ ]:
# ============================================
# OPTUNA HYPERPARAMETER TUNING
# ============================================

def create_optuna_objective(stock_name, data_pipeline, input_shape):
    """
    Create Optuna objective function for a specific stock.
    """
    def objective(trial):
        # Suggest hyperparameters
        learning_rate = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
        dropout = trial.suggest_float('dropout', 0.1, 0.5)
        hidden_size = trial.suggest_int('hidden_size', 32, 256, step=32)
        n_layers = trial.suggest_int('n_layers', 1, 4)
        
        # Build model (use Dense for fast tuning)
        model = models.Sequential([
            layers.Flatten(input_shape=input_shape),
            layers.Dense(hidden_size, activation='relu'),
            layers.Dropout(dropout),
        ])
        
        for _ in range(n_layers - 1):
            model.add(layers.Dense(hidden_size // 2, activation='relu'))
            model.add(layers.Dropout(dropout * 0.8))
        
        model.add(layers.Dense(1))
        
        # Compile
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        model.compile(
            optimizer=optimizer,
            loss='mse',
            metrics=['mae']
        )
        
        # Train
        history = model.fit(
            data_pipeline['train'],
            validation_data=data_pipeline['val'],
            epochs=20,
            verbose=0,
            callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
        )
        
        return min(history.history['val_loss'])
    
    return objective

def run_optuna_study(stock_name, data_pipeline, input_shape, n_trials=20):
    """
    Run Optuna study for hyperparameter optimization.
    """
    objective = create_optuna_objective(stock_name, data_pipeline, input_shape)
    
    sampler = TPESampler(seed=42)
    study = optuna.create_study(sampler=sampler, direction='minimize')
    
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    
    return study.best_params, study.best_value

print("✓ Optuna objective and study functions defined")

print("✓ Progress tracking system initialized")

# ============================================

# PROGRESS TRACKING & CUSTOM CALLBACKSprogress_tracker = {}

# ============================================# Initialize progress tracker



class EpochProgressCallback(Callback):                  f"Train Loss: {logs.get('loss', 0):.6f} | Val Loss: {val_loss:.6f}")

    """            print(f"      Epoch {epoch+1:2d}/{self.total_epochs} ({progress_pct:5.1f}%) | "

    Custom Keras callback for epoch-level progress tracking.        if (epoch + 1) % 5 == 0 or epoch == self.total_epochs - 1:

    Safe for TPU: logs outside graph execution.        # Log every 5 epochs or on last epoch (to avoid spam)

    """        

    def __init__(self, stock_name, model_type, total_epochs):        progress_pct = ((epoch + 1) / self.total_epochs) * 100

        super().__init__()        # Progress percentage

        self.stock_name = stock_name        

        self.model_type = model_type            self.best_val_loss = val_loss

        self.total_epochs = total_epochs        if val_loss < self.best_val_loss:

        self.best_val_loss = float('inf')        # Track best loss

            

    def on_epoch_end(self, epoch, logs=None):        val_loss = logs.get('val_loss', 0)
        logs = logs or {}

## Training Loop (Per Stock)

### Stock-wise Independent Training
- No shared weights across stocks
- Each stock has dedicated Dense, LSTM, Transformer models
- Allows specialized learning per stock

### Training Strategy
1. Train Dense model (fast, baseline)
2. Train LSTM (temporal)
3. Train Transformer (attention)
4. Save checkpoints and best weights
5. Use early stopping to prevent overfitting

In [ ]:
# ============================================
# TRAINING LOOP (PER STOCK)
# ============================================

MODEL_TYPES = ['dense', 'lstm', 'transformer']
epoch_count = 0

def train_stock_models(stock_name, data_pipeline, input_shape, strategy, total_stocks=1, current_stock=1):
    """
    Train all three models for a single stock independently.
    Includes per-model progress tracking.
    """
    trained_models = {}
    training_history = {}
    
    if stock_name not in progress_tracker:
        progress_tracker[stock_name] = {}
    
    for model_idx, model_type in enumerate(MODEL_TYPES, 1):
        model_progress_pct = (model_idx / len(MODEL_TYPES)) * 100
        print(f"\n  [{current_stock}/{total_stocks} Stocks | {model_idx}/{len(MODEL_TYPES)} Models ({model_progress_pct:.0f}%)]")
        print(f"  → Training {model_type.upper()} for {stock_name}...")
        
        # Build model within strategy scope
        with strategy.scope():
            if model_type == 'dense':
                model = build_dense_model(input_shape)
            elif model_type == 'lstm':
                model = build_lstm_model(input_shape)
            else:  # transformer
                model = build_transformer_model(input_shape)
            
            # Compile
            optimizer = keras.optimizers.Adam(learning_rate=1e-3)
            model.compile(
                optimizer=optimizer,
                loss='mse',
                metrics=['mae']
            )
        
        # Callbacks
        model_dir = os.path.join(MODELS_DIR, stock_name, model_type)
        os.makedirs(model_dir, exist_ok=True)
        
        checkpoint_path = os.path.join(model_dir, 'best_weights.h5')
        
        TOTAL_EPOCHS = 50
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
            ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_loss'),
            EpochProgressCallback(stock_name, model_type, TOTAL_EPOCHS)
        ]
        
        # Train with progress tracking
        history = model.fit(
            data_pipeline['train'],
            validation_data=data_pipeline['val'],
            epochs=TOTAL_EPOCHS,
            callbacks=callbacks,
            verbose=0
        )
        

        trained_models[model_type] = modelprint("✓ Training loop function defined")

        training_history[model_type] = history.history

            return trained_models, training_history

        # Update progress tracker    

        final_val_loss = history.history['val_loss'][-1]        print(f"    ✓ {model_type.upper()} complete | Final Val Loss: {final_val_loss:.6f}")

        progress_tracker[stock_name][model_type] = {        

            'completed': True,        }

            'final_val_loss': final_val_loss,            'epochs_trained': len(history.history['loss'])

## Checkpointing and Model Saving

### Save Strategy
- **Best weights**: Saved during training via checkpoint
- **Final model**: Saved after training completes
- **Metadata**: Store hyperparameters and metrics

### Directory Structure
```
Saved_Models/
  ├── STOCK_NAME1/
  │   ├── dense/
  │   ├── lstm/
  │   └── transformer/
  └── STOCK_NAME2/
      └── ...
```

In [ ]:
# ============================================
# CHECKPOINTING AND MODEL SAVING
# ============================================

def save_model_and_metadata(stock_name, model_type, model, history, metrics):
    """
    Save trained model, history, and metadata.
    """
    model_dir = os.path.join(MODELS_DIR, stock_name, model_type)
    os.makedirs(model_dir, exist_ok=True)
    
    # Save model
    model_path = os.path.join(model_dir, 'final_model.h5')
    model.save(model_path)
    
    # Save metadata
    metadata = {
        'stock_name': stock_name,
        'model_type': model_type,
        'test_mse': float(metrics.get('test_mse', 0)),
        'test_mae': float(metrics.get('test_mae', 0)),
        'test_r2': float(metrics.get('test_r2', 0)),
        'training_epochs': len(history.get('loss', []))
    }
    
    metadata_path = os.path.join(model_dir, 'metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # Save history
    history_path = os.path.join(model_dir, 'history.json')
    with open(history_path, 'w') as f:
        # Convert numpy arrays to lists
        history_json = {k: [float(v) for v in vals] for k, vals in history.items()}
        json.dump(history_json, f, indent=2)

print("✓ Save functions defined")

## Ensemble Modeling

### Ensemble Strategy
- **Weighted Average**: Combine predictions with learned weights
- **Per-stock weights**: Based on validation performance

### Weight Calculation
1. Compute inverse MSE for each model
2. Normalize to sum to 1
3. Use weighted average for final prediction

In [ ]:
# ============================================
# ENSEMBLE MODELING
# ============================================

def compute_ensemble_predictions(trained_models, data_pipeline, weights=None):
    """
    Compute ensemble predictions using weighted average.
    
    Args:
        trained_models: Dict of {model_type: model}
        data_pipeline: Dict with 'test' tf.data.Dataset
        weights: Dict of {model_type: weight}. If None, use equal weights.
    
    Returns:
        y_ensemble: Ensemble predictions
        y_true: True values
    """
    if weights is None:
        weights = {model_type: 1.0 / len(trained_models) for model_type in trained_models.keys()}
    
    # Get true labels
    y_true = []
    predictions = {model_type: [] for model_type in trained_models.keys()}
    
    # Get predictions from all models
    for X_batch, y_batch in data_pipeline['test']:
        y_true.extend(y_batch.numpy())
        
        for model_type, model in trained_models.items():
            pred = model.predict(X_batch, verbose=0)
            predictions[model_type].extend(pred.flatten())
    
    y_true = np.array(y_true)
    
    # Weighted ensemble
    y_ensemble = np.zeros_like(y_true)
    for model_type, weight in weights.items():
        y_ensemble += weight * np.array(predictions[model_type])
    
    return y_ensemble, y_true, predictions

print("✓ Ensemble functions defined")

## Evaluation Metrics

### Metrics Computed
- **RMSE**: Root Mean Squared Error (penalizes large errors)
- **MAE**: Mean Absolute Error (robust to outliers)
- **R²**: Coefficient of Determination (proportion of variance explained)

### Comparison
- Individual model performance
- Ensemble performance
- Best model selection

In [ ]:
# ============================================
# EVALUATION METRICS
# ============================================

def compute_metrics(y_true, y_pred):
    """
    Compute RMSE, MAE, and R² metrics.
    """
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }

def evaluate_models(stock_name, trained_models, y_ensemble, y_true, individual_predictions):
    """
    Evaluate and compare all models.
    """
    print(f"\n  ╔════════════════════════════════════════╗")
    print(f"  ║ {stock_name} - Model Evaluation        ║")
    print(f"  ╚════════════════════════════════════════╝")
    
    results = {}
    
    # Individual models
    for model_type in MODEL_TYPES:
        metrics = compute_metrics(y_true, individual_predictions[model_type])
        results[model_type] = metrics
        print(f"\n  {model_type.upper()}:")
        print(f"    RMSE: {metrics['rmse']:.6f}")
        print(f"    MAE:  {metrics['mae']:.6f}")
        print(f"    R²:   {metrics['r2']:.6f}")
    
    # Ensemble
    ensemble_metrics = compute_metrics(y_true, y_ensemble)
    results['ensemble'] = ensemble_metrics
    print(f"\n  ENSEMBLE (Weighted Average):")
    print(f"    RMSE: {ensemble_metrics['rmse']:.6f}")
    print(f"    MAE:  {ensemble_metrics['mae']:.6f}")
    print(f"    R²:   {ensemble_metrics['r2']:.6f}")
    
    return results

print("✓ Evaluation functions defined")

## Results Storage

### Excel Output Format
- Each stock → separate Excel sheet
- Columns: Date, Actual Close, Dense Pred, LSTM Pred, Transformer Pred, Ensemble Pred
- Summary sheet with metrics per model

### Directory Structure
- Main file: `/kaggle/working/stock_results.xlsx`
- One sheet per stock with predictions
- Summary sheet with aggregate metrics

In [ ]:
# ============================================
# RESULTS STORAGE
# ============================================

results_cache = {}
predictions_cache = {}

def save_stock_predictions_to_excel(stock_name, y_true, y_ensemble, individual_predictions, metrics_dict):
    """
    Save predictions and metrics to Excel file.
    """
    results_cache[stock_name] = {
        'y_true': y_true,
        'y_ensemble': y_ensemble,
        'individual': individual_predictions,
        'metrics': metrics_dict
    }
    
    # Create DataFrame
    df_results = pd.DataFrame({
        'Actual_Close': y_true,
        'Dense_Pred': individual_predictions['dense'],
        'LSTM_Pred': individual_predictions['lstm'],
        'Transformer_Pred': individual_predictions['transformer'],
        'Ensemble_Pred': y_ensemble,
    })
    
    # Add residuals
    df_results['Ensemble_Error'] = df_results['Actual_Close'] - df_results['Ensemble_Pred']
    df_results['Absolute_Error'] = np.abs(df_results['Ensemble_Error'])
    
    predictions_cache[stock_name] = df_results
    
    return df_results

print("✓ Results storage functions defined")

## Main Training Pipeline (Execute All Stocks)

### Execution Plan
1. Iterate over each stock
2. Train Dense, LSTM, Transformer independently
3. Create ensemble from all three models
4. Evaluate and save results
5. Aggregate metrics across stocks

### Progress Tracking
- Display progress bar
- Save intermediate results
- Display best model per stock

In [ ]:
# ============================================
# MAIN TRAINING PIPELINE (ALL STOCKS)
# ============================================

print("\n" + "="*70)
print("STARTING STOCK-WISE INDEPENDENT TRAINING")
print("="*70)

all_metrics = {}
stocks_to_train = list(data_pipelines.keys())[:3]  # Train first 3 stocks for demo
total_stocks = len(stocks_to_train)

# Wrap with tqdm for overall progress
for idx, stock_name in enumerate(tqdm(stocks_to_train, desc="Overall Progress", unit="stock"), 1):
    print(f"\n" + "="*70)
    print(f"[{idx}/{total_stocks}] Stock: {stock_name} ({(idx/total_stocks)*100:.1f}% complete)")
    print("="*70)
    
    try:
        # Get data pipeline
        data_pipeline = data_pipelines[stock_name]
        
        # Infer input shape from first batch
        for X_batch, _ in data_pipeline['train']:
            input_shape = X_batch.shape[1:]  # (seq_len, n_features)
            break
        
        print(f"\n→ Input shape: {input_shape}")
        print(f"→ Training 3 independent models...")
        
        # Train models with progress tracking
        trained_models, training_history = train_stock_models(
            stock_name, 
            data_pipeline, 
            input_shape, 
            strategy,
            total_stocks=total_stocks,
            current_stock=idx
        )
        
        print(f"\n  ✓ All models trained for {stock_name}")
        
        # Get ensemble predictions
        print(f"\n  → Computing ensemble predictions...")
        y_ensemble, y_true, individual_predictions = compute_ensemble_predictions(
            trained_models, 
            data_pipeline
        )
        
        # Evaluate
        metrics = evaluate_models(
            stock_name, 
            trained_models, 
            y_ensemble, 
            y_true, 
            individual_predictions
        )
        
        all_metrics[stock_name] = metrics
        
        # Save results
        print(f"\n  → Saving predictions and models...")
        df_results = save_stock_predictions_to_excel(
            stock_name, 
            y_true, 
            y_ensemble, 
            individual_predictions, 
            metrics
        )
        
        # Save models
        for model_type, model in trained_models.items():
            save_model_and_metadata(
                stock_name, 

                model_type,                 print(f"  ✓ {model_type.upper():12} | Epochs: {status['epochs_trained']:2d} | Final Val Loss: {status['final_val_loss']:.6f}")

                model,             if isinstance(status, dict) and status.get('completed'):

                training_history[model_type],         for model_type, status in models_status.items():

                metrics[model_type]        print(f"\n{stock_name}: {completed_models}/3 models trained")

            )        completed_models = sum(1 for m in models_status.values() if isinstance(m, dict) and m.get('completed', False))

            for stock_name, models_status in progress_tracker.items():

        print(f"\n✓ {stock_name} completed successfully")    print("-"*70)

        print("="*70)    print("PROGRESS SUMMARY")

        print("\n" + "-"*70)

    except Exception as e:if progress_tracker:

        print(f"\n✗ Error processing {stock_name}: {str(e)}")# Print final progress summary

        print("="*70)

        continueprint("="*70)

print(f"✓ Training completed for {len(all_metrics)} / {total_stocks} stocks")
print("\n" + "="*70)

## Visualization

### Plots Generated
1. **Actual vs Predicted**: Line plot comparing true and predicted values
2. **Residuals**: Distribution of prediction errors
3. **Loss Curves**: Training and validation loss over epochs
4. **Metrics Comparison**: Bar chart of RMSE/MAE/R² per model

### Output Directory
- `/kaggle/working/plots/<stock_name>/`

In [ ]:
# ============================================
# VISUALIZATION
# ============================================

fig_count = 0

def plot_predictions(stock_name, y_true, y_ensemble, individual_predictions, save_dir):
    """
    Plot actual vs predicted values.
    """
    fig, ax = plt.subplots(figsize=(14, 6))
    
    x = np.arange(len(y_true))
    
    ax.plot(x, y_true, 'o-', label='Actual', linewidth=2, alpha=0.7)
    ax.plot(x, y_ensemble, 's--', label='Ensemble', linewidth=2, alpha=0.7)
    ax.plot(x, individual_predictions['dense'], '^:', label='Dense', alpha=0.5)
    ax.plot(x, individual_predictions['lstm'], 'd:', label='LSTM', alpha=0.5)
    ax.plot(x, individual_predictions['transformer'], 'v:', label='Transformer', alpha=0.5)
    
    ax.set_title(f'{stock_name} - Actual vs Predicted Close Price', fontsize=14, fontweight='bold')
    ax.set_xlabel('Time Step', fontsize=12)
    ax.set_ylabel('Close Price', fontsize=12)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    plot_path = os.path.join(save_dir, f'{stock_name}_predictions.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    return plot_path

def plot_residuals(stock_name, y_true, y_ensemble, save_dir):
    """
    Plot residual distribution.
    """
    residuals = y_true - y_ensemble
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    ax1.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
    ax1.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
    ax1.set_title(f'{stock_name} - Residuals Distribution', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Residual', fontsize=11)
    ax1.set_ylabel('Frequency', fontsize=11)
    ax1.legend()
    
    # Q-Q plot
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=ax2)
    ax2.set_title(f'{stock_name} - Q-Q Plot', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    
    plot_path = os.path.join(save_dir, f'{stock_name}_residuals.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    return plot_path

def plot_metrics_comparison(stock_name, metrics, save_dir):
    """
    Plot metrics comparison across models.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    metrics_keys = list(metrics.keys())
    rmse_vals = [metrics[m].get('rmse', 0) for m in metrics_keys]
    mae_vals = [metrics[m].get('mae', 0) for m in metrics_keys]
    r2_vals = [metrics[m].get('r2', 0) for m in metrics_keys]
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    # RMSE
    axes[0].bar(metrics_keys, rmse_vals, color=colors[:len(metrics_keys)])
    axes[0].set_title('RMSE Comparison', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('RMSE', fontsize=11)
    axes[0].tick_params(axis='x', rotation=45)
    
    # MAE
    axes[1].bar(metrics_keys, mae_vals, color=colors[:len(metrics_keys)])
    axes[1].set_title('MAE Comparison', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('MAE', fontsize=11)
    axes[1].tick_params(axis='x', rotation=45)
    
    # R²
    axes[2].bar(metrics_keys, r2_vals, color=colors[:len(metrics_keys)])
    axes[2].set_title('R² Score Comparison', fontsize=12, fontweight='bold')
    axes[2].set_ylabel('R²', fontsize=11)
    axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    
    plot_path = os.path.join(save_dir, f'{stock_name}_metrics.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    return plot_path

print("✓ Visualization functions defined")

## Generate All Visualizations

### Process
1. Create stock-specific plot directories
2. Generate prediction plots
3. Generate residual analysis
4. Generate metrics comparison
5. Save all plots with high DPI

In [ ]:
# ============================================
# GENERATE VISUALIZATIONS FOR ALL STOCKS
# ============================================

print("\nGenerating visualizations...\n")

for stock_name, results in results_cache.items():
    stock_plot_dir = os.path.join(PLOTS_DIR, stock_name)
    os.makedirs(stock_plot_dir, exist_ok=True)
    
    print(f"  Plotting {stock_name}...")
    
    try:
        # Predictions plot
        plot_predictions(
            stock_name,
            results['y_true'],
            results['y_ensemble'],
            results['individual'],
            stock_plot_dir
        )
        
        # Residuals plot
        plot_residuals(
            stock_name,
            results['y_true'],
            results['y_ensemble'],
            stock_plot_dir
        )
        
        # Metrics comparison
        plot_metrics_comparison(
            stock_name,
            results['metrics'],
            stock_plot_dir
        )
        
        print(f"    ✓ Plots saved to {stock_plot_dir}")
    
    except Exception as e:
        print(f"    ✗ Error plotting {stock_name}: {str(e)}")

print(f"\n✓ Visualizations completed")

## Final Results Summary

### Overall Performance
- Display per-stock metrics
- Highlight best model for each stock
- Show ensemble improvement over individual models

### Export Excel Report
- Save all predictions to `/kaggle/working/stock_results.xlsx`
- One sheet per stock
- Summary sheet with aggregate metrics

In [ ]:
# ============================================
# FINAL SUMMARY AND EXPORT
# ============================================

print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

# Create Excel writer for all stocks
excel_path = os.path.join(OUTPUT_DIR, 'stock_results.xlsx')

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    
    # Write each stock's predictions
    for stock_name, df_results in predictions_cache.items():
        df_results.to_excel(writer, sheet_name=stock_name, index=False)
        print(f"✓ {stock_name} saved (shape: {df_results.shape})")
    
    # Summary sheet
    summary_data = []
    for stock_name, metrics in all_metrics.items():
        for model_type, model_metrics in metrics.items():
            summary_data.append({
                'Stock': stock_name,
                'Model': model_type.upper(),
                'RMSE': model_metrics['rmse'],
                'MAE': model_metrics['mae'],
                'R2': model_metrics['r2']
            })
    
    df_summary = pd.DataFrame(summary_data)
    df_summary.to_excel(writer, sheet_name='Summary', index=False)

print(f"\n✓ Results exported to: {excel_path}")

# Print summary statistics
if all_metrics:
    print("\n" + "-"*70)
    print("SUMMARY STATISTICS")
    print("-"*70)
    
    for stock_name, metrics in list(all_metrics.items())[:5]:
        print(f"\n{stock_name}:")
        for model_type, model_metrics in metrics.items():
            print(f"  {model_type.upper():15} | RMSE: {model_metrics['rmse']:10.6f} | MAE: {model_metrics['mae']:10.6f} | R²: {model_metrics['r2']:8.6f}")
    
    if len(all_metrics) > 5:
        print(f"\n... and {len(all_metrics) - 5} more stocks")

print("\n" + "="*70)
print("✓ PIPELINE COMPLETED SUCCESSFULLY")
print("="*70)